In [2]:
import numpy as np
import xarray as xr
from tqdm.notebook import trange, tqdm
import glob
import matplotlib as mpl
import matplotlib.pyplot as plt
#import seaborn as sns
import datetime
import time
import os,sys
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from contextlib import redirect_stdout

import sys
sys.path.insert(1, '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt')

from nn_functions.constants import *
import nn_functions.diagnostic_functions as diag
import nn_functions.data_formatting as dfmt
import nn_functions.postprocessing_functions as pp
#from nn_functions.constants import *

In [3]:
mod_size = 'small'
TS_opt = 'extrap'
norm_method = 'std'
exp_name = 'NEMO_grid_v0_slope_front'
exp_name = 'newbasic2'
seed_nb = 1
print('Set options')

np.random.seed(seed_nb)
tf.random.set_seed(seed_nb)

Set options


In [4]:
nemo_run = 'OPM026'

In [5]:
#inputpath_data_nn = '/bettik/burgardc/DATA/NN_PARAM/interim/INPUT_DATA/'
inputpath_data_nn = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/'


# Filepath for normalised inputs
filepath_data_AIAI = "/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/AIAI_data/"
filepath_norm_data = filepath_data_AIAI + 'normed_data'
inputpath_data = filepath_norm_data + '_' 
# Filepath for outputs 
outputpath_nn_models = filepath_data_AIAI + 'NN_models/'
#outputpath_doc = '/bettik/burgardc/SCRIPTS/basal_melt_neural_networks/custom_doc/experiments/'
#outputpath_doc = 

#inputpath_CV = '/bettik/burgardc/DATA/NN_PARAM/interim/INPUT_DATA/'
inputpath_CV = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/INPUT_DATA/'
#outputpath_nn_models = '/bettik/burgardc/DATA/NN_PARAM/interim/NN_MODELS/experiments/'
outputpath_nn_models = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/NN_MODELS/'

data_train_orig_norm = xr.open_dataset(inputpath_data + 'train_data_wholedataset.nc')
data_val_orig_norm = xr.open_dataset(inputpath_data + 'val_data_wholedataset.nc') 



In [6]:
if exp_name == 'NEMO_grid_v0_slope_lat_lon':
    var_list = ['distances_GL', 'distances_OO', 'distances_OC', 'temperature_prop', 'salinity_prop', \
                'corrected_isdraft', 'bathymetry', 'slope_is_lon', 'slope_is_lat', 'slope_ba_lon', \
                'slope_ba_lat', 'mean_T', 'mean_S', 'std_T', 'std_S', 'melt_ice_per_yr']
elif exp_name == 'NEMO_grid_v0_slope_front':
    var_list = ['distances_GL', 'distances_OO', 'distances_OC', 'temperature_prop', 'salinity_prop', \
                'corrected_isdraft', 'bathymetry', 'slope_is_across_front', 'slope_is_towards_front', \
                'slope_ba_across_front', 'slope_ba_towards_front', 'mean_T', 'mean_S', 'std_T', 'std_S', 'melt_ice_per_yr']

In [ ]:
#inputpath_data='/bettik/burgardc/DATA/NN_PARAM/interim/SMITH_'+nemo_run+'/'
inputpath_data='/bettik/burgardc/DATA/NN_PARAM/interim/nemo_5km_'+nemo_run+'/'
inputpath_data='/bettik/burgardc/DATA/BASAL_MELT_PARAM/interim/NEMO_eORCA025.L121_'+nemo_run+'_ANT_STEREO/'
# Required subfiles in data folder are
# inputpath_data+'other_mask_vars_Ant_stereo.nc'
# inputpath_data+'corrected_draft_bathy_isf.nc'
# inputpath_data+'isfdraft_conc_Ant_stereo.nc'
# But I think these are actually all things needed for the Smith data and not for my purposes?

### use any model from CV over time
#outputpath_melt_nn = '/bettik/burgardc/DATA/NN_PARAM/processed/MELT_RATE/SMITH_'+nemo_run+'/'
outputpath_melt_nn = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/processed/MELT_RATE/nemo_5km_'+nemo_run+'/'
#path_model = '/bettik/burgardc/DATA/NN_PARAM/interim/NN_MODELS/experiments/WHOLE/'
path_model = '/bettik/ockendeh/SCRIPTS/simpleNN_basal_melt/data/interim/NN_MODELS/'